In [1]:
##------------Imports, paths------------

import os
import zipfile
import sqlite3
import requests
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

nrows=500000
source_url = "https://www.fs.usda.gov/rds/archive/products/RDS-2013-0009.7/RDS-2013-0009.7_Data_Format4_SQLITE.zip"    
local_dir = "wildfire_7th_edition_data"
os.makedirs(local_dir, exist_ok=True)
zip_path = os.path.join(local_dir, "us_wildfires_v7.zip")

In [2]:
##------------Data download - run one time only------------

if 1==1:
    print("📡 Initializing browser emulation handshake with U.S. Forest Service...")
    # 1. BYPASS 403 FORBIDDEN: Fake a verified user desktop connection header
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }
    # 2. STREAM SECURE TRANSFER: Download block-by-block to keep memory low
    if not os.path.exists(zip_path):
        print("📥 Streaming 7th Edition database (~219MB compressed package)... Please wait.")
        with requests.get(source_url, headers=headers, stream=True) as response:
            response.raise_for_status()
            with open(zip_path, 'wb') as f:
                for chunk in response.iter_content(chunk_size=16384):
                    if chunk:
                        f.write(chunk)
        print("✅ Download finished completely.")
        
    # 3. EXTRACTION SEQUENCE
    print("🔓 Unpacking compressed archive contents...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(local_dir)

if 1==1: 
    # 4. READ metadata file    
    data_dir = local_dir+'/Data/'
    df_meta = pd.read_csv(data_dir+'_variable_descriptions.csv', keep_default_na=False)
    print(df_meta)

    # 5. READ data file
    db_path = data_dir+'FPA_FOD_20260615.sqlite'
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # 6. List all available tables within the master catalog schema
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    print("📁 Available Tables in Database:")
    tables = cursor.fetchall()
    for t in tables:
        print(f"   ↳ {t[0]}")

    # 7. Inspect column headers and data types inside the main 'Fires' table
    cursor.execute("PRAGMA table_info(Fires);")
    print("\n📊 Column Schema Details for 'Fires' Table:")
    columns = cursor.fetchall()
    for col in columns:
        # col[1] = Column Name, col[2] = Data Type (TEXT, REAL, INT)
        print(f"   ↳ Header: {col[1]:<20} | Type: {col[2]}")


📡 Initializing browser emulation handshake with U.S. Forest Service...
📥 Streaming 7th Edition database (~219MB compressed package)... Please wait.
✅ Download finished completely.
🔓 Unpacking compressed archive contents...
                                              Filename            Variable  \
0                       \Data\FPA_FOD_20260615 [Fires]              FOD_ID   
1                       \Data\FPA_FOD_20260615 [Fires]              FPA_ID   
2                       \Data\FPA_FOD_20260615 [Fires]  SOURCE_SYSTEM_TYPE   
3                       \Data\FPA_FOD_20260615 [Fires]  SOURCE_SYSTEM_TYPE   
4                       \Data\FPA_FOD_20260615 [Fires]  SOURCE_SYSTEM_TYPE   
..                                                 ...                 ...   
202  \Data\FPA_FOD_20260615 [NWCG_UnitIdActive_2020...             Country   
203  \Data\FPA_FOD_20260615 [NWCG_UnitIdActive_2020...             Country   
204  \Data\FPA_FOD_20260615 [NWCG_UnitIdActive_2020...               State 

In [3]:
##------------Load data------------

if 1==1:
    # The 7th Edition utilizes updated National Wildfire Coordinating Group (NWCG) columns
    query = f"""
    SELECT 
        OBJECTID,
        FIRE_NAME,
        COMPLEX_NAME,
        FIRE_YEAR,
        DISCOVERY_DATE,
        DISCOVERY_DOY,
        DISCOVERY_TIME,
        CONT_DATE,
        CONT_DOY,
        CONT_TIME,
        FIRE_SIZE,
        FIRE_SIZE_CLASS,
        LATITUDE,
        LONGITUDE,
        OWNER_DESCR,
        STATE,
        COUNTY,
        FIPS_NAME,
        FIPS_CODE,
        NWCG_REPORTING_AGENCY,
        SOURCE_SYSTEM_TYPE,
        NWCG_CAUSE_CLASSIFICATION,
        NWCG_GENERAL_CAUSE
    FROM Fires
    WHERE CONT_DATE IS NOT NULL 
      AND DISCOVERY_DATE IS NOT NULL
    LIMIT {nrows};
    """
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    print(f"🎉 Success! 7th Edition Data Matrix populated smoothly. Shape: {df.shape}")
    print("\n--- Sample Telemetry Output ---")
df

🎉 Success! 7th Edition Data Matrix populated smoothly. Shape: (500000, 23)

--- Sample Telemetry Output ---


,OBJECTID,FIRE_NAME,COMPLEX_NAME,FIRE_YEAR,DISCOVERY_DATE,DISCOVERY_DOY,DISCOVERY_TIME,CONT_DATE,CONT_DOY,CONT_TIME,...,LONGITUDE,OWNER_DESCR,STATE,COUNTY,FIPS_NAME,FIPS_CODE,NWCG_REPORTING_AGENCY,SOURCE_SYSTEM_TYPE,NWCG_CAUSE_CLASSIFICATION,NWCG_GENERAL_CAUSE
0,1,FOUNTAIN,None,2005,2453403.5,33,1300,2453403.5,33,1730,...,-121.005833,USFS,CA,63,Plumas County,06063,FS,FED,Human,Power generation/transmission/distribution
1,2,PIGEON,None,2004,2453137.5,133,0845,2453137.5,133,1530,...,-120.404444,USFS,CA,61,Placer County,06061,FS,FED,Natural,Natural
2,3,SLACK,None,2004,2453156.5,152,1921,2453156.5,152,2024,...,-120.735556,STATE OR PRIVATE,CA,17,El Dorado County,06017,FS,FED,Human,Debris and open burning
3,4,DEER,None,2004,2453184.5,180,1600,2453189.5,185,1400,...,-119.913333,USFS,CA,3,Alpine County,06003,FS,FED,Natural,Natural
4,5,STEVENOT,None,2004,2453184.5,180,1600,2453189.5,185,1200,...,-119.933056,USFS,CA,3,Alpine County,06003,FS,FED,Natural,Natural
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
499995,741793,None,None,1993,2449166.5,179,None,2449167.5,180,1010,...,-81.158333,MISSING/NOT SPECIFIED,SC,32,Lexington County,45063,ST/C&L,NONFED,Missing data/not specified/undetermined,Missing data/not specified/undetermined
499996,741794,None,None,1993,2449168.5,181,None,2449168.5,181,None,...,-81.425000,MISSING/NOT SPECIFIED,SC,32,Lexington County,45063,ST/C&L,NONFED,Missing data/not specified/undetermined,Missing data/not specified/undetermined
499997,741795,None,None,1993,2449168.5,181,None,2449168.5,181,None,...,-81.325000,MISSING/NOT SPECIFIED,SC,32,Lexington County,45063,ST/C&L,NONFED,Missing data/not specified/undetermined,Missing data/not specified/undetermined
499998,741796,None,None,1992,2448815.5,194,None,2448815.5,194,1215,...,-80.962500,MISSING/NOT SPECIFIED,SC,38,Orangeburg County,45075,ST/C&L,NONFED,Missing data/not specified/undetermined,Missing data/not specified/undetermined
